# PDBe

Download and filter PDB files.

## Environment setup

Run the next cell once per fresh kernel to install notebook dependencies.
If you install packages in the active kernel, restart the kernel and rerun all cells.

In [ ]:
# Cloud and local notebooks: install required runtime dependencies.
%pip install -q protein-quest[nb]

## Download PDB files from PDBe

In [1]:
from pathlib import Path

from protein_quest.pdbe.fetch import fetch

In [2]:
ids = [
    "4NDY",  # structure of uniprot A8MT69
    "4DRA",  # structure of uniprot A8MT69
    "1XWH",  # structure of uniprot O43918
    "8WAS",  # structure for which there is no single pdb file
]

In [3]:
save_dir = Path("pdb_files")

In [4]:
files = await fetch(ids, save_dir)
files

{'4NDY': PosixPath('pdb_files/4ndy_updated.cif.gz'),
 '4DRA': PosixPath('pdb_files/4dra_updated.cif.gz'),
 '1XWH': PosixPath('pdb_files/1xwh_updated.cif.gz'),
 '8WAS': PosixPath('pdb_files/8was_updated.cif.gz')}

In [5]:
!ls pdb_files

1xwh_updated.cif.gz  4ndy_updated.cif.gz
4dra_updated.cif.gz  8was_updated.cif.gz


## Prepare pdb file for haddock3 or powerfit

Haddock3 and powerfit like to have a file with a single protein in it called chain "A". The protein-quest package can help to create such a file.

In [6]:
from protein_quest.structure.chains import write_single_chain_structure_file

In [11]:
# A8MT69	4NDY	B/D/H/L/M/N/U/V/W/X=8-81
# above is the identifier, chain and position info for the structure from https://www.uniprot.org/uniprotkb/A8MT69/entry#structure
output_4dny_file = write_single_chain_structure_file(
    input_file=save_dir / "4ndy_updated.cif.gz", chain2keep="B", output_dir=save_dir
)
output_4dny_file

PosixPath('pdb_files/4ndy_updated_B2A.cif.gz')

In [12]:
# A8MT69	4DRA	E/F/G/H=1-81
output_4dra_file = write_single_chain_structure_file(
    input_file=save_dir / "4dra_updated.cif.gz", chain2keep="E", output_dir=save_dir
)
output_4dra_file

PosixPath('pdb_files/4dra_updated_E2A.cif.gz')

In [13]:
output_1xwh_file = write_single_chain_structure_file(
    input_file=save_dir / "1xwh_updated.cif.gz",
    chain2keep="A",
    output_dir=save_dir,
)
output_1xwh_file

PosixPath('pdb_files/1xwh_updated_A2A.cif.gz')

In [14]:
# O00268      │ 8WAS    │ D/d=1-1085
output_8was_file = write_single_chain_structure_file(
    input_file=save_dir / "8was_updated.cif.gz",
    chain2keep="D",
    output_dir=save_dir,
)
output_8was_file

PosixPath('pdb_files/8was_updated_D2A.cif.gz')

In [15]:
!ls -sh -1 pdb_files/

total 4.8M
388K 1xwh_updated.cif.gz
388K 1xwh_updated_A2A.cif.gz
284K 4dra_updated.cif.gz
 28K 4dra_updated_E2A.cif.gz
836K 4ndy_updated.cif.gz
 32K 4ndy_updated_B2A.cif.gz
 28K 4ndy_updated_E2A.cif.gz
2.9M 8was_updated.cif.gz
 32K 8was_updated_D2A.cif.gz


## Visualize a structure with Mol*

Use [molviewspec](https://molstar.org/mol-view-spec/) to visualize one of the structures.

Cell output is absent to keep notebook size small, please run yourself to see visualization.

In [ ]:
from molviewspec import create_builder, molstar_notebook

structure_file = Path("pdb_files/8was_updated_D2A.cif.gz")

builder = create_builder()
builder.download(url=structure_file.name).parse(format="mmcif").model_structure().component().representation().color(
    color="blue"
)

molstar_notebook(state=builder.get_state(), data={structure_file.name: structure_file.read_bytes()})